In [ ]:
# Setup for Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 6 - Ensemble Learning

## Mục tiêu:
- Sử dụng Random Forest và XGBoost để tối ưu độ chính xác.
- So sánh hiệu suất giữa Decision Tree đơn lẻ, Random Forest và XGBoost.
- Giải thích khái niệm Voting và Boosting.
- Dataset sử dụng: Iris dataset.

## 1. Lý thuyết: Voting và Boosting

### Voting (Bỏ phiếu)
Voting là một phương pháp Ensemble Learning kết hợp dự đoán từ nhiều mô hình (base estimators) khác nhau để đưa ra kết quả cuối cùng. Có hai loại chính:
- **Hard Voting:** Mỗi mô hình đưa ra một dự đoán (phiếu bầu) cho một lớp (class). Lớp nhận được nhiều phiếu bầu nhất sẽ là kết quả cuối cùng theo nguyên tắc đa số thắng thiểu số.
- **Soft Voting:** Mỗi mô hình trả về xác suất dự đoán cho từng lớp. Kết quả cuối cùng là lớp có giá trị trung bình xác suất cao nhất. Phương pháp này thường cho hiệu suất tốt hơn vì nó tính đến sự "chắc chắn" của các mô hình.

### Boosting (Tăng cường)
Boosting là một phương pháp Ensemble kết hợp các mô hình yếu (weak learners) thành một mô hình mạnh (strong learner) bằng cách huấn luyện chúng một cách **tuần tự**.
Thay vì huấn luyện độc lập như trong Bagging (VD: Random Forest), mỗi mô hình mới trong Boosting sẽ tập trung sửa chữa những sai lầm của mô hình trước đó. Quá trình này được thực hiện bằng cách tăng trọng số cho các điểm dữ liệu bị phân loại sai (như trong AdaBoost) hoặc tối ưu hóa hàm mất mát (như trong Gradient Boosting, XGBoost).

## 2. Chuẩn bị dữ liệu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Tải dữ liệu Iris
iris = load_iris()
X = iris.data
y = iris.target

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Kích thước tập huấn luyện:", X_train.shape)
print("Kích thước tập kiểm tra:", X_test.shape)

## 3. Huấn luyện Decision Tree (Mô hình cơ sở)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Khởi tạo và huấn luyện mô hình Decision Tree
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred_tree = tree_clf.predict(X_test)
acc_tree = accuracy_score(y_test, y_pred_tree)

print(f"Độ chính xác của Decision Tree: {acc_tree:.4f}")

## 4. Huấn luyện Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Khởi tạo và huấn luyện mô hình Random Forest (100 cây)
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred_rf = rf_clf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Độ chính xác của Random Forest: {acc_rf:.4f}")

## 5. Huấn luyện XGBoost

*Lưu ý: Nếu bạn chưa cài đặt `xgboost`, hãy chạy lệnh `!pip install xgboost` trong một cell mới trước khi chạy đoạn code dưới đây.*

In [ ]:
# !pip install xgboost # Bỏ comment nếu cần cài đặt thư viện
import xgboost as xgb

# Khởi tạo và huấn luyện mô hình XGBoost
# eval_metric='mlogloss' được sử dụng cho bài toán phân loại đa lớp để tránh cảnh báo (warning)
xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
xgb_clf.fit(X_train, y_train)

# Dự đoán và đánh giá
y_pred_xgb = xgb_clf.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

print(f"Độ chính xác của XGBoost: {acc_xgb:.4f}")

## 6. So sánh hiệu suất

In [ ]:
models = ['Decision Tree', 'Random Forest', 'XGBoost']
accuracies = [acc_tree, acc_rf, acc_xgb]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=['#1f77b4', '#2ca02c', '#ff7f0e'])

# Hiển thị giá trị trên các cột
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f'{yval:.4f}', ha='center', va='bottom', fontweight='bold')

plt.ylim(0.85, 1.05) # Cắt một phần trục y để dễ nhìn thấy sự khác biệt nhỏ
plt.ylabel('Độ chính xác (Accuracy)')
plt.title('So sánh độ chính xác của các mô hình trên tập Iris')
plt.show()